In [1]:
RUN_OFFICIAL_TEST = False


# 14: Official test final evaluation

Este notebook usa el test oficial FD001. Ejecutar solo despues de cerrar la seleccion final por validacion.

Debe correrse ultimo. Mientras `RUN_OFFICIAL_TEST = False`, no entrena ni evalua el test oficial.


In [2]:
from collections import OrderedDict
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RESULTS_DIR = PROJECT_ROOT / "results" / "FD001"
FIGURES_DIR = PROJECT_ROOT / "figures"
PREDICTIONS_DIR = PROJECT_ROOT / "predictions"
for path in [RESULTS_DIR, FIGURES_DIR, PREDICTIONS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

DATA_DIR = PROJECT_ROOT / "CMAPSSData"
EVAL_SIZE = 0.2
VALIDATION_RANDOM_STATE = 42
CUT_RULS = (20, 50, 80, 110, 140)
BASE_WINDOW_SIZE = 30
BASE_RUL_CAP = 125

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)


In [3]:
from src.fd001_experiment_utils import (
    FlexibleMLPRegressor,
    available_boosting_factories,
    fit_predict_model,
    lgbm_factory,
)


In [4]:
from src.fd001_experiment_utils import (
    final_model_from_row,
    parse_rul_cap,
    parse_window_size,
    weights_from_scheme,
)


In [5]:
if not RUN_OFFICIAL_TEST:
    print("RUN_OFFICIAL_TEST = False. No se entrena ni evalua test oficial.")
else:
    ranking_path = RESULTS_DIR / "fd001_final_validation_ranking.csv"
    if not ranking_path.exists():
        raise FileNotFoundError("Primero ejecutar el notebook 13 para generar fd001_final_validation_ranking.csv")

    ranking = pd.read_csv(ranking_path)
    selected = ranking.iloc[0]
    model_name = selected["model_name"]
    representation = str(selected["representation"])
    window_size = parse_window_size(selected)
    rul_cap = parse_rul_cap(selected.get("rul_cap", BASE_RUL_CAP))
    sample_weight_scheme = selected.get("sample_weight_scheme", "none")

    print("Modelo final elegido por validacion artificial:")
    display(selected)

    if representation == "current_cycle":
        from src.preprocessed_FD001 import prepare_fd001_current_cycle_full_train_for_test
        final_prepared = prepare_fd001_current_cycle_full_train_for_test(
            data_dir=DATA_DIR,
            max_rul=rul_cap,
        )
    else:
        final_prepared = prepare_fd001_temporal_full_train_for_test(
            data_dir=DATA_DIR,
            max_rul=rul_cap,
            window_size=window_size,
        )

    model = final_model_from_row(selected)
    sample_weight = weights_from_scheme(final_prepared["y_train_raw"], sample_weight_scheme)
    if sample_weight is None:
        model.fit(final_prepared["X_train"], final_prepared["y_train"])
    else:
        model.fit(final_prepared["X_train"], final_prepared["y_train"], sample_weight=sample_weight)

    official_predictions = official_test_prediction_frame(
        final_prepared,
        model,
        model_name=model_name,
        representation=representation,
    )[["unit", "y_true_rul_raw", "y_pred_rul", "error", "abs_error"]]
    official_predictions.to_csv(PREDICTIONS_DIR / "fd001_best_model_predictions.csv", index=False)

    metrics = regression_metrics(
        official_predictions["y_true_rul_raw"],
        official_predictions["y_pred_rul"],
    )
    official_metrics = pd.DataFrame([{
        "model_name": model_name,
        "representation": representation,
        "window_size": window_size,
        "rul_cap": "None" if rul_cap is None else rul_cap,
        "sample_weight_scheme": sample_weight_scheme,
        "n_test": len(official_predictions),
        **metrics,
    }])
    official_metrics.to_csv(RESULTS_DIR / "fd001_official_test_metrics.csv", index=False)
    display(official_metrics)
    display(official_predictions.head())


RUN_OFFICIAL_TEST = False. No se entrena ni evalua test oficial.
